# Pairs fOU 02 — the thesis pipeline on construction-tied spreads (Binance)

Protocol: `discussions/Pairs_fOU/003_claude_pf02_basis_preregistration.md`.

**SP**: perpetual − spot (≈30 assets). **PQ**: front quarterly − perpetual (where quarterlies exist). β = 1 by construction; ADF (1%) replaces Engle–Granger; otherwise the PF01 fOU engine unchanged. Taker costs: spot 11 bp, futures 6 bp per side per leg; real funding on the perp leg.

Long runtime (1–3 h). Per-asset results are checkpointed in `02/parts/`, so a re-run resumes.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import fou_pairs as fp, basis_fou as bf
for m in (fp,bf):importlib.reload(m)
assert bf.VERSION=='PF02-0.1.0' and fp.VERSION.startswith('PF01-0.1.1')
spec=fp.Spec();prm=bf.Params()
SYNTHETIC=os.getenv('PF02_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='pf02-');OUT=Path(tmp.name)/'02';spec=fp.Spec(paths=800,structural_paths=400,n_boot=500)
else:
    OUT=ROOT/'notebooks'/'Pairs_fOU'/'02'
(OUT/'parts').mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_basis_fou.py'),str(ROOT/'tests'/'test_fou_pairs.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-800:]);print(chk.stderr[-500:]);assert chk.returncode==0,'tests failed'
print(json.dumps(bf.registration(),indent=1)[:1500])

In [ ]:
def load_inputs(A):
    """Returns {'SP': (panel, funding_y, funding_x, cy, cx), 'PQ': ...} for one asset."""
    if SYNTHETIC:
        P,f=bf.synthetic_basis(n_days=12,seed=hash(A)%100,kappa=0.08,sd_bp=25)
        P.columns=[f'PERP_{A}',f'SPOT_{A}']
        return {'SP':(P,f,None,prm.fut_cost_bp,prm.spot_cost_bp)}
    sym=f'{A}USDT';out={}
    pp=BIN/f'{sym}_um_1m.parquet';sp=BIN/'basis'/'spot'/f'{sym}_spot_1m.parquet';fu=BIN/'basis'/'funding'/f'{sym}_funding.parquet'
    if not pp.exists():return out
    perp=fp.log_close_5m(pd.read_parquet(pp,columns=['open_time','close']))
    fund=bf.funding_series(pd.read_parquet(fu)) if fu.exists() else None
    if sp.exists():
        spot=fp.log_close_5m(pd.read_parquet(sp,columns=['open_time','close']))
        out['SP']=(fp.panel({f'PERP_{A}':perp,f'SPOT_{A}':spot}),fund,None,prm.fut_cost_bp,prm.spot_cost_bp)
    qfiles=sorted((BIN/'basis'/'quarterly').glob(f'{sym}_*_1m.parquet'))
    if qfiles:
        con={}
        for q in qfiles:
            e=pd.Timestamp('20'+q.name.split('_')[1],tz='UTC')+pd.Timedelta(hours=8)
            con[e]=fp.log_close_5m(pd.read_parquet(q,columns=['open_time','close']))
        front=bf.front_series(con,prm.roll_days_before_expiry)
        out['PQ']=(fp.panel({f'QTR_{A}':front,f'PERP_{A}':perp}).dropna(how='all'),None,fund,prm.fut_cost_bp,prm.fut_cost_bp)
    return out
if not SYNTHETIC:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
ASSETS=['AAA','BBB'] if SYNTHETIC else list(prm.universe)
print('assets:',ASSETS)

## Run (checkpointed per asset and family)

In [ ]:
T0=time.time();scale=[]
for A in ASSETS:
    inp=load_inputs(A)
    if not inp:print(A,'no perp file - skipped');continue
    for fam,(P,fy_,fx_,cy,cx) in inp.items():
        y,x=P.columns[0],P.columns[1]
        part=OUT/'parts'/f'{fam}_{A}'
        s=bf.deviation_scale(P,y,x);scale.append({'family':fam,'asset':A,'median_7d_sd_bp':float(s.median()),
             'p90_7d_sd_bp':float(s.quantile(.9)),'bars':int(P.notna().all(axis=1).sum())})
        if (part.with_suffix('.fits.parquet')).exists():
            print(f'{fam} {A}: cached');continue
        cache={};TR=[];FC=[];FI=[]
        for mode in ('fou','ou','zscore'):
            tr,fc,fi=fp.run_pair(P,y,x,spec,mode=mode,screen_cache=cache,screen_fn=bf.screen_basis,
                                 funding_y=fy_,funding_x=fx_,cost_y_bp=cy,cost_x_bp=cx,label=f'{fam}_{A}')
            TR.append(tr);FC.append(fc);FI.append(fi)
        tr=pd.concat([t for t in TR if len(t)],ignore_index=True) if any(len(t) for t in TR) else pd.DataFrame()
        fc=pd.concat([f for f in FC if len(f)],ignore_index=True) if any(len(f) for f in FC) else pd.DataFrame()
        fi=pd.concat(FI,ignore_index=True)
        if len(tr):tr.to_parquet(part.with_suffix('.trades.parquet'),index=False)
        if len(fc):fc.drop(columns=[c for c in fc.columns if fc[c].dtype==object and c not in('pair','mode')],errors='ignore').to_parquet(part.with_suffix('.fc.parquet'),index=False)
        fi.drop(columns=['resid'],errors='ignore').to_parquet(part.with_suffix('.fits.parquet'),index=False)
        print(f'{fam} {A}: trades={len(tr)} eligible fou days={int(fi[fi["mode"]=="fou"].eligible.sum())} | {time.time()-T0:6.0f}s',flush=True)
SCALE=pd.DataFrame(scale);display(SCALE.round(2))

In [ ]:
def rd(sfx):
    fs=sorted((OUT/'parts').glob(f'*.{sfx}.parquet'))
    return pd.concat([pd.read_parquet(f) for f in fs],ignore_index=True) if fs else pd.DataFrame()
trades,fcs,fits=rd('trades'),rd('fc'),rd('fits')
for d in (trades,fits,fcs):
    if len(d):d['family']=d['pair'].str.split('_').str[0]
TS=pd.Timestamp(spec.test_start,tz='UTC')
print('Formation funnel (fOU):');display(fits[fits['mode']=='fou'].groupby(['family','reason']).size().unstack(fill_value=0))
if len(trades):trades['period']=np.where(trades.entry_bar>=TS,'test','dev')
rows=[]
for fam in sorted(trades.family.unique()) if len(trades) else []:
    for mode in ('fou','ou','zscore'):
        for per in ('dev','test'):
            t=trades[(trades.family==fam)&(trades['mode']==mode)&(trades.period==per)]
            rows.append(fp.trade_summary(t,spec,f'{fam}|{mode}|{per}|all'))
            if mode=='fou':
                for side,g in t.groupby('side'):rows.append(fp.trade_summary(g,spec,f'{fam}|{mode}|{per}|side={side}'))
                for a,g in t.groupby('pair'):rows.append(fp.trade_summary(g,spec,f'{fam}|{mode}|{per}|{a}'))
summ=pd.DataFrame(rows)
display(summ[~summ.label.str.contains(r'\|(?:SP|PQ)_')].round(2))
print('Per asset (fOU, TEST):');display(summ[summ.label.str.contains('fou\\|test\\|(?:SP|PQ)_')].round(2))
if len(trades):print('Mean funding and cost per trade (bp of gross notional):');display(trades.groupby(['family','mode'])[['gross_bp','funding_bp','cost_bp','net_bp']].mean().round(2))

In [ ]:
cal=[]
for fam in sorted(fcs.family.unique()) if len(fcs) else []:
    for mode in ('fou','ou'):
        for per in ('dev','test'):
            f=fcs[(fcs.family==fam)&(fcs['mode']==mode)];f=f[(f.signal_bar>=TS)==(per=='test')]
            cal.append({'family':fam,'mode':mode,'period':per,**fp.calibration(f)})
cal=pd.DataFrame(cal);display(cal.round(4))
def decide(summ):
    if SYNTHETIC:return {'verdict':'SYNTHETIC_NO_MARKET_INFERENCE'}
    out={}
    for fam in ('SP','PQ'):
        r=summ[summ.label==f'{fam}|fou|test|all']
        if r.empty or not r.iloc[0].get('trades',0):out[fam]={'verdict':'NO_TRADES'};continue
        r=r.iloc[0]
        v=('NET_EDGE_CANDIDATE' if r.net_lo>0 else 'GROSS_EDGE_ONLY' if r.gross_lo>0 else
           'NO_EDGE' if (r.net_hi<0) else 'INCONCLUSIVE')
        out[fam]={'verdict':v,'trades':int(r.trades),'gross':[r.gross_bp,r.gross_lo,r.gross_hi],'net':[r.net_bp,r.net_lo,r.net_hi]}
    out['primary']='SP';out['status']=prm.status;return out
dec=decide(summ);print(json.dumps(dec,indent=2,default=float))
summ.to_csv(OUT/'02_summary.csv',index=False);cal.to_csv(OUT/'02_calibration.csv',index=False);SCALE.to_csv(OUT/'02_basis_scale.csv',index=False)
fits.to_csv(OUT/'02_formation.csv',index=False,float_format='%.6g')
if len(trades):trades.to_csv(OUT/'02_trades.csv.gz',index=False,float_format='%.6g')
(OUT/'02_decision.json').write_text(json.dumps(dec,indent=2,default=float),encoding='utf-8')
(OUT/'02_manifest.json').write_text(json.dumps({'registration':bf.registration(),'synthetic':SYNTHETIC,'runtime_s':time.time()-T0},indent=2,default=str),encoding='utf-8')
print('saved to',OUT)